# Deep learning: frozen embeddings, fine-tuned DistilBERT, zero-shot LLM (notebook 03)

Session 7 order: first freeze a pretrained encoder and train only a small head on its
embeddings (cheap, cached), then fine-tune the whole encoder (expensive). Last, a zero-shot
language model (Session 8) answers "is this ad fraudulent?" with no training at all. Each model is scored
once on test per (model, seed) on the same five splits as notebook 02; comparing and plotting
the saved results is notebook 04. Heavy cells need a GPU and the `torch`, `transformers` and
`sentence-transformers` packages (plus `captum` for the explanation); the run cells are skipped unless `ON_GPU_SERVER = True`
(set it on the server only).

Run time: one DistilBERT run took about 7 minutes on one GPU at batch size 128 (measured, seed 0,
5 epochs); the embedding heads take seconds once the embeddings are cached.

In [1]:
import sys, os; os.chdir(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else None; sys.path.insert(0, "."); import warnings
import importlib.util

from modules import data_helper as dh
from modules import features as ft
from modules import deep_runs as dr

DATA = "../data/fake_job_postings.csv"

In [2]:
SEED = 0
SEEDS = [0, 1, 2, 3, 4]
BERT_SEEDS = SEEDS  # seeds with existing result files are skipped
OUT_DEEP = "outputs/deep"

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
BERT_MODEL = "distilbert-base-uncased"
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
EPOCHS, BATCH_SIZE, MAX_LEN, LR = 5, 128, 256, 2e-5
LOG_EVERY = 10  # print the loss every this many steps
DEVICE = "cuda:2"  # which GPU to use; the server is shared, so pick an emptier one (nvidia-smi), e.g. "cuda:1"

# SAFETY: the embedding and fine-tuning cells are heavy GPU work. They run only when this flag is
# True, and it must stay False on the laptop. Set it to True on the GPU server, nowhere else.
ON_GPU_SERVER = True

# Libraries present? (a missing one would make the heavy cells fail, not skip)
HAVE_TORCH = all(importlib.util.find_spec(m)
                 for m in ("torch", "transformers", "sentence_transformers", "captum"))
HAVE_EMBEDDINGS = os.path.exists("features/" + EMBED_MODEL.replace("/", "_") + "_emb.npy")

# Heavy cells need the explicit opt-in AND the libraries; the sklearn heads are light (CPU) and
# only need the cached embeddings.
RUN_EMBED = RUN_BERT = RUN_LLM = ON_GPU_SERVER and HAVE_TORCH
RUN_HEADS = RUN_EMBED or HAVE_EMBEDDINGS
print({"ON_GPU_SERVER": ON_GPU_SERVER, "torch stack": HAVE_TORCH, "cached embeddings": HAVE_EMBEDDINGS})

dh.seed_everything(SEED)

{'ON_GPU_SERVER': True, 'torch stack': True, 'cached embeddings': True}


## Data and splits

The same cleaned frame and the same saved splits as the classical notebook, so every model
is scored on identical rows.

In [3]:
df = ft.add_engineered_features(dh.clean(dh.load_raw(DATA)))
print("rows:", len(df), "| fraud:", int(df["fraudulent"].sum()))

rows: 17599 | fraud: 856


## Step 1: frozen sentence embeddings

`all-MiniLM-L6-v2` turns each ad (the joined `text` column) into one 384-dimensional vector.
Nothing is trained here. The vectors are cached in `features/` keyed by `job_id`, so this
cell is a no-op the second time.

In [4]:
if RUN_EMBED:
    # Encode every ad once; the result is cached in features/.
    dr.run_embed(df, DEVICE, EMBED_MODEL)
else:
    print("skipped: set ON_GPU_SERVER = True (GPU server only) and install torch + sentence-transformers")

embeddings: (17599, 384) ready in 0.1s


## Step 2: logistic regression on the frozen embeddings

A logistic regression (C picked on validation PR-AUC) trained on the cached vectors with CPU only.
It is the same classifier as the TF-IDF logistic regression in notebook 02, so the pair compares
the text representation alone. Run for all five seeds; the threshold is picked on validation and
the test rows are scored once.

In [5]:
if RUN_HEADS:
    dr.run_heads(df, SEEDS, OUT_DEEP, EMBED_MODEL)
else:
    print("skipped: no cached embeddings here")

embeddings: (17599, 384)
seed 0:
  train n=10559 positives= 514 rate=0.0487
  val   n= 3520 positives= 171 rate=0.0486
  test  n= 3520 positives= 171 rate=0.0486
skip emb_logreg seed 0: result files already exist
seed 1:
  train n=10559 positives= 514 rate=0.0487
  val   n= 3520 positives= 171 rate=0.0486
  test  n= 3520 positives= 171 rate=0.0486
skip emb_logreg seed 1: result files already exist
seed 2:
  train n=10559 positives= 514 rate=0.0487
  val   n= 3520 positives= 171 rate=0.0486
  test  n= 3520 positives= 171 rate=0.0486
skip emb_logreg seed 2: result files already exist
seed 3:
  train n=10559 positives= 514 rate=0.0487
  val   n= 3520 positives= 171 rate=0.0486
  test  n= 3520 positives= 171 rate=0.0486
skip emb_logreg seed 3: result files already exist
seed 4:
  train n=10559 positives= 514 rate=0.0487
  val   n= 3520 positives= 171 rate=0.0486
  test  n= 3520 positives= 171 rate=0.0486
skip emb_logreg seed 4: result files already exist


## Step 3: fine-tune DistilBERT

The whole encoder is trained with a linear classification layer on top: AdamW, learning rate
2e-5 with warm-up and linear decay, square-root class weights in the loss, bf16 autocast on
CUDA, and early stopping on validation PR-AUC (the best epoch's weights are restored). Texts
are truncated to 256 tokens. `BERT_SEEDS` lists the seeds to fine-tune; finished ones are skipped.

Model explanation: right after seed 0 is trained, Integrated Gradients scores how much each token
pushed the model towards fraud, for 10 seed-0 test ads (4 caught frauds, 3 missed frauds, 3 false
alarms). They are saved next to the results and shown in notebook 04.

In [6]:
if RUN_BERT:
    dr.run_bert(df, BERT_SEEDS, OUT_DEEP, DEVICE, BERT_MODEL, epochs=EPOCHS,
                batch_size=BATCH_SIZE, max_len=MAX_LEN, lr=LR, log_every=LOG_EVERY)
else:
    print("skipped: set ON_GPU_SERVER = True (GPU server only) and install torch + transformers")

seed 0:
  train n=10559 positives= 514 rate=0.0487
  val   n= 3520 positives= 171 rate=0.0486
  test  n= 3520 positives= 171 rate=0.0486
loading distilbert-base-uncased (downloads on the first run)...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenising...
fine-tuning...
  step 0/83  loss 0.6763
  step 10/83  loss 0.6273
  step 20/83  loss 0.5192
  step 30/83  loss 0.4268
  step 40/83  loss 0.4041
  step 50/83  loss 0.4656
  step 60/83  loss 0.4128
  step 70/83  loss 0.4049
  step 80/83  loss 0.1150
epoch 1/5: train loss 0.4173, val PR-AUC 0.604, 46s
  step 0/83  loss 0.2934
  step 10/83  loss 0.1340
  step 20/83  loss 0.1007
  step 30/83  loss 0.1100
  step 40/83  loss 0.3481
  step 50/83  loss 0.0984
  step 60/83  loss 0.1856
  step 70/83  loss 0.0621
  step 80/83  loss 0.1798
epoch 2/5: train loss 0.1654, val PR-AUC 0.809, 57s
  step 0/83  loss 0.0739
  step 10/83  loss 0.0537
  step 20/83  loss 0.3159
  step 30/83  loss 0.0561
  step 40/83  loss 0.0836
  step 50/83  loss 0.0415
  step 60/83  loss 0.0286
  step 70/83  loss 0.1750
  step 80/83  loss 0.1131
epoch 3/5: train loss 0.0903, val PR-AUC 0.846, 56s
  step 0/83  loss 0.0235
  step 10/83  loss 0.0127
  step 20/83  loss 0.0368
  step 30/83  loss 0.3412
  step 40/83 

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenising...
fine-tuning...
  step 0/83  loss 0.7000
  step 10/83  loss 0.6302
  step 20/83  loss 0.6510
  step 30/83  loss 0.3672
  step 40/83  loss 0.4118
  step 50/83  loss 0.2898
  step 60/83  loss 0.3914
  step 70/83  loss 0.3195
  step 80/83  loss 0.2173
epoch 1/5: train loss 0.4139, val PR-AUC 0.682, 46s
  step 0/83  loss 0.2759
  step 10/83  loss 0.1534
  step 20/83  loss 0.1534
  step 30/83  loss 0.2138
  step 40/83  loss 0.1544
  step 50/83  loss 0.1317
  step 60/83  loss 0.2106
  step 70/83  loss 0.0959
  step 80/83  loss 0.0775
epoch 2/5: train loss 0.1632, val PR-AUC 0.798, 54s
  step 0/83  loss 0.0961
  step 10/83  loss 0.1161
  step 20/83  loss 0.0320
  step 30/83  loss 0.0651
  step 40/83  loss 0.0205
  step 50/83  loss 0.0754
  step 60/83  loss 0.0451
  step 70/83  loss 0.2287
  step 80/83  loss 0.0156
epoch 3/5: train loss 0.0911, val PR-AUC 0.812, 56s
  step 0/83  loss 0.0163
  step 10/83  loss 0.0379
  step 20/83  loss 0.1187
  step 30/83  loss 0.0193
  step 40/83 

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenising...
fine-tuning...
  step 0/83  loss 0.6317
  step 10/83  loss 0.5844
  step 20/83  loss 0.3989
  step 30/83  loss 0.3440
  step 40/83  loss 0.3867
  step 50/83  loss 0.2991
  step 60/83  loss 0.3305
  step 70/83  loss 0.2938
  step 80/83  loss 0.2011
epoch 1/5: train loss 0.3990, val PR-AUC 0.544, 55s
  step 0/83  loss 0.1722
  step 10/83  loss 0.2125
  step 20/83  loss 0.2946
  step 30/83  loss 0.0605
  step 40/83  loss 0.1123
  step 50/83  loss 0.0567
  step 60/83  loss 0.0700
  step 70/83  loss 0.2064
  step 80/83  loss 0.1376
epoch 2/5: train loss 0.1589, val PR-AUC 0.830, 47s
  step 0/83  loss 0.0944
  step 10/83  loss 0.0563
  step 20/83  loss 0.0363
  step 30/83  loss 0.1099
  step 40/83  loss 0.2067
  step 50/83  loss 0.1170
  step 60/83  loss 0.1559
  step 70/83  loss 0.0326
  step 80/83  loss 0.0724
epoch 3/5: train loss 0.0982, val PR-AUC 0.833, 42s
  step 0/83  loss 0.0834
  step 10/83  loss 0.0192
  step 20/83  loss 0.0910
  step 30/83  loss 0.0137
  step 40/83 

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenising...
fine-tuning...
  step 0/83  loss 0.6804
  step 10/83  loss 0.6472
  step 20/83  loss 0.5388
  step 30/83  loss 0.3655
  step 40/83  loss 0.2617
  step 50/83  loss 0.3048
  step 60/83  loss 0.2341
  step 70/83  loss 0.3157
  step 80/83  loss 0.4665
epoch 1/5: train loss 0.4199, val PR-AUC 0.612, 42s
  step 0/83  loss 0.2245
  step 10/83  loss 0.1374
  step 20/83  loss 0.1024
  step 30/83  loss 0.2304
  step 40/83  loss 0.0914
  step 50/83  loss 0.0986
  step 60/83  loss 0.0926
  step 70/83  loss 0.0626
  step 80/83  loss 0.0718
epoch 2/5: train loss 0.1571, val PR-AUC 0.801, 45s
  step 0/83  loss 0.2818
  step 10/83  loss 0.0330
  step 20/83  loss 0.1001
  step 30/83  loss 0.0387
  step 40/83  loss 0.1294
  step 50/83  loss 0.2471
  step 60/83  loss 0.0665
  step 70/83  loss 0.0551
  step 80/83  loss 0.0314
epoch 3/5: train loss 0.0760, val PR-AUC 0.831, 50s
  step 0/83  loss 0.0273
  step 10/83  loss 0.0157
  step 20/83  loss 0.0099
  step 30/83  loss 0.0090
  step 40/83 

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenising...
fine-tuning...
  step 0/83  loss 0.7165
  step 10/83  loss 0.6731
  step 20/83  loss 0.5332
  step 30/83  loss 0.5216
  step 40/83  loss 0.3414
  step 50/83  loss 0.5287
  step 60/83  loss 0.2497
  step 70/83  loss 0.2615
  step 80/83  loss 0.2941
epoch 1/5: train loss 0.4213, val PR-AUC 0.574, 52s
  step 0/83  loss 0.1378
  step 10/83  loss 0.1833
  step 20/83  loss 0.3062
  step 30/83  loss 0.1798
  step 40/83  loss 0.3373
  step 50/83  loss 0.2195
  step 60/83  loss 0.2032
  step 70/83  loss 0.2195
  step 80/83  loss 0.2009
epoch 2/5: train loss 0.1683, val PR-AUC 0.796, 44s
  step 0/83  loss 0.0437
  step 10/83  loss 0.0444
  step 20/83  loss 0.0622
  step 30/83  loss 0.0554
  step 40/83  loss 0.0655
  step 50/83  loss 0.0569
  step 60/83  loss 0.0944
  step 70/83  loss 0.3208
  step 80/83  loss 0.0339
epoch 3/5: train loss 0.0920, val PR-AUC 0.813, 45s
  step 0/83  loss 0.0712
  step 10/83  loss 0.0625
  step 20/83  loss 0.0383
  step 30/83  loss 0.0321
  step 40/83 

## Step 4: zero-shot language model

`Qwen2.5-1.5B-Instruct` reads each ad (cut to 512 tokens) in a short Yes/No prompt; the fraud
score is its probability of answering "Yes" (renormalised over Yes and No). Nothing is trained,
so every ad is scored once and the scores are cached in `features/`; each seed then only picks
its threshold on validation and scores its test rows, like every other model.

In [7]:
if RUN_LLM:
    dr.run_zero_shot(df, SEEDS, OUT_DEEP, DEVICE, LLM_MODEL)
else:
    print("skipped: set ON_GPU_SERVER = True (GPU server only) and install torch + transformers")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  zero-shot 16/17599
  zero-shot 816/17599
  zero-shot 1616/17599
  zero-shot 2416/17599
  zero-shot 3216/17599
  zero-shot 4016/17599
  zero-shot 4816/17599
  zero-shot 5616/17599
  zero-shot 6416/17599
  zero-shot 7216/17599
  zero-shot 8016/17599
  zero-shot 8816/17599
  zero-shot 9616/17599
  zero-shot 10416/17599
  zero-shot 11216/17599
  zero-shot 12016/17599
  zero-shot 12816/17599
  zero-shot 13616/17599
  zero-shot 14416/17599
  zero-shot 15216/17599
  zero-shot 16016/17599
  zero-shot 16816/17599
zeroshot_Qwen2.5-1.5B-Instruct seed 0: val PR-AUC 0.098, test PR-AUC 0.075
zeroshot_Qwen2.5-1.5B-Instruct seed 1: val PR-AUC 0.065, test PR-AUC 0.077
zeroshot_Qwen2.5-1.5B-Instruct seed 2: val PR-AUC 0.119, test PR-AUC 0.083
zeroshot_Qwen2.5-1.5B-Instruct seed 3: val PR-AUC 0.111, test PR-AUC 0.065
zeroshot_Qwen2.5-1.5B-Instruct seed 4: val PR-AUC 0.114, test PR-AUC 0.089
